# Solving the Freise model
We seek to solve the Freise finite-size diffuse-layer model for a symmetric 1:1 electrolyte with different cation and anion volume factors:
$$
\boxed{
\begin{aligned}
    \text{Freise}&\begin{cases}
    \varepsilon_{0}\varepsilon_{r}\dfrac{\mathrm{d}^{2}\phi}{\mathrm{d}x^{2}}
    &=
    \dfrac{2 e_{0} N_{\mathrm{A}}c_{\mathrm{b}}
    \sinh\left(\dfrac{e_{0}\phi}{k_{\mathrm{B}}T}\right)}
    {1 + r_{0}^{3}N_{\mathrm{A}}c_{\mathrm{b}}
    \left[
    \gamma_{\mathrm{c}}\left(e^{-e_{0}\phi/k_{\mathrm{B}}T}-1\right)
    +
    \gamma_{\mathrm{a}}\left(e^{+e_{0}\phi/k_{\mathrm{B}}T}-1\right)
    \right]},
    &0 < x < +\infty  \\
    \phi\left(x\right) &= \phi_{\mathrm{M}} - \phi_{\mathrm{pzc}},
    &x = 0 \\
    \phi\left(x\right) &= 0,
    &x = +\infty .
    \end{cases}
\end{aligned}
}
$$

Compared with the Bikerman model, the Freise form allows the finite-size correction to distinguish between cations and anions through the two relative volume factors $\gamma_{\mathrm{c}}$ and $\gamma_{\mathrm{a}}$.  When $\gamma_{\mathrm{c}}=\gamma_{\mathrm{a}}=1$, the denominator reduces to the symmetric Bikerman/Freise steric denominator.


# Parametric setup  ###

### The following lines of code set up the parameters and fundamental constants of the Freise model


In [ ]:
from dataclasses import dataclass
import math

# Fundamental constants
e = 1.602176634e-19       # C
kB = 1.380649e-23         # J/K
eps0 = 8.8541878128e-12   # F/m
NA = 6.02214076e23        # 1/mol


@dataclass
class Freise_SI_Params:
    r0: float                  # Reference lattice / molecular length, m

    gamma_c: float = 1.0       # Cation relative volume factor
    gamma_a: float = 1.0       # Anion relative volume factor

    eps_S: float = 78.5
    c_bulk: float = 1000.0
    T: float = 298.15

    tol: float = 1e-12
    maxiter: int = 200


def compute_beta(p: Freise_SI_Params) -> float:
    return 1.0 / (kB * p.T)


def compute_n_bulk(p: Freise_SI_Params) -> float:
    """
    Number density of each ion species for a symmetric 1:1 electrolyte.

    If c_bulk is the salt concentration in mol/m^3, then
    n_bulk = N_A c_bulk.
    """
    return NA * p.c_bulk


def compute_volume_parameter(p: Freise_SI_Params) -> float:
    """
    Freise volume parameter for the symmetric 1:1 implementation:

        alpha = r0^3 n_bulk

    The finite-size denominator becomes

        D(U) = 1 + alpha [gamma_c (exp(-U) - 1) + gamma_a (exp(+U) - 1)].

    If gamma_c = gamma_a = 1, this reduces to

        D(U) = 1 + 4 r0^3 n_bulk sinh^2(U/2),

    i.e. the symmetric Bikerman denominator used in the previous notebook.
    """
    n_bulk = compute_n_bulk(p)
    return p.r0**3 * n_bulk


def compute_lambda_D(p: Freise_SI_Params) -> float:
    """
    Debye length for a symmetric 1:1 electrolyte.

    This is still useful as a reference length scale.
    """
    return math.sqrt(
        p.eps_S * eps0 * kB * p.T /
        (2.0 * e**2 * NA * p.c_bulk)
    )


def validate_params_si(p: Freise_SI_Params) -> None:
    if p.T <= 0:
        raise ValueError("T must be > 0")
    if p.eps_S <= 0:
        raise ValueError("eps_S must be > 0")
    if p.c_bulk <= 0:
        raise ValueError("c_bulk must be > 0")
    if p.r0 <= 0:
        raise ValueError("r0 must be > 0")
    if p.gamma_c < 0:
        raise ValueError("gamma_c must be >= 0")
    if p.gamma_a < 0:
        raise ValueError("gamma_a must be >= 0")

    # Freise packing constraint for the two mobile ions in this implementation.
    # If explicit solvent is added later, include its volume fraction here as well.
    alpha = compute_volume_parameter(p)
    gamma_D = 1.0 - alpha * (p.gamma_c + p.gamma_a)
    if gamma_D < -1e-12 or gamma_D > 1.0 + 1e-12:
        raise ValueError(
            "Freise bulk-packing constraint failed: "
            "0 <= 1 - r0^3 n_bulk (gamma_c + gamma_a) <= 1. "
            f"Current value is {gamma_D:.4g}."
        )


# Helper functions

### For stability, we define the following voltage conversion functions

$$
\begin{align*}
V_{\mathrm{T}} &= \frac{k_{\mathrm{B}}T}{e_{0}},
&U &= \frac{\phi}{V_{\mathrm{T}}},
&\phi &= U V_{\mathrm{T}} .
\end{align*}
$$

Here $U$ is the dimensionless electrostatic potential.


In [ ]:
def thermal_voltage(p: Freise_SI_Params) -> float:
    return kB * p.T / e


def V_to_U(p: Freise_SI_Params, V: float) -> float:
    return V / thermal_voltage(p)


def U_to_V(p: Freise_SI_Params, U: float) -> float:
    return U * thermal_voltage(p)


# Surface Free Charge Density and Double Layer Capacitance

## Exact Freise first integral

For the Freise model, define

$$
U = \frac{e_0(\phi_M-\phi_{\mathrm{pzc}})}{k_BT},
\qquad
 t = e^U,
\qquad
 n_0=N_Ac_b .
$$

The finite-size denominator is

$$
D(U)=1+r_0^3n_0\left[\gamma_c(e^{-U}-1)+\gamma_a(e^U-1)\right].
$$

After the substitution $t=e^U$, the denominator becomes the quadratic form

$$
D(U)=\frac{a_0t^2+a_1t+a_2}{t},
$$

with

$$
a_0=r_0^3n_0\gamma_a,
\qquad
 a_1=1-r_0^3n_0(\gamma_a+\gamma_c),
\qquad
 a_2=r_0^3n_0\gamma_c .
$$

The first integral needed for the surface charge is

$$
L(U)
=
\int_1^{e^U}
\frac{t^2-1}{t(a_0t^2+a_1t+a_2)}\,dt .
$$

The code below evaluates this integral analytically with the required coefficient case splits: $a_0=0$, $a_2=0$, $a_1=0$, the pure-polynomial edge cases, and the general quadratic case using the sign of $4a_0a_2-a_1^2$.

## Surface free charge density

$$
\sigma_{\mathrm{free}}^{\left(\mathrm{F}\right)}
=
\operatorname{sgn}(\phi_M-\phi_{\mathrm{pzc}})
\sqrt{\frac{2\varepsilon_0\varepsilon_S n_0}{\beta}}
\sqrt{L(U)},
\qquad
\beta=\frac{1}{k_BT} .
$$

## Double-layer capacitance

The differential capacitance follows from

$$
C_{\mathrm{F}}
=
\frac{\partial \sigma_{\mathrm{free}}^{\left(\mathrm{F}\right)}}
{\partial (\phi_M-\phi_{\mathrm{pzc}})} .
$$

For nonzero surface potential, the expression used here is

$$
C_{\mathrm{F}}
=
e_0\sqrt{\frac{\beta n_0\varepsilon_0\varepsilon_S}{2}}
\operatorname{sgn}(\phi_0)
\frac{t^2-1}{(a_0t^2+a_1t+a_2)\sqrt{L(U)}} .
$$

At the PZC, the limiting Gouy--Chapman value is recovered:

$$
C_{\mathrm{F}}(0)
=
\sqrt{2\varepsilon_0\varepsilon_S\beta e_0^2 n_0}.
$$


In [ ]:
import math
import numpy as np


def freise_quadratic_coefficients(p: Freise_SI_Params):
    """
    Coefficients of the Freise quadratic after t = exp(U):

        D(U) = [a0 t^2 + a1 t + a2] / t.

    with
        a0 = alpha gamma_a,
        a1 = 1 - alpha (gamma_a + gamma_c),
        a2 = alpha gamma_c,
        alpha = r0^3 n0.
    """
    validate_params_si(p)
    alpha = compute_volume_parameter(p)
    a0 = alpha * p.gamma_a
    a2 = alpha * p.gamma_c
    a1 = 1.0 - alpha * (p.gamma_a + p.gamma_c)
    return float(a0), float(a1), float(a2)


def freise_D_from_U(p: Freise_SI_Params, U):
    validate_params_si(p)
    U_arr = np.asarray(U, dtype=float)
    alpha = compute_volume_parameter(p)

    return (
        1.0
        + alpha * (
            p.gamma_c * np.expm1(-U_arr)
            + p.gamma_a * np.expm1(+U_arr)
        )
    )


def _safe_log_abs(x):
    return np.log(np.abs(x))


def _freise_quadratic_integral_K(t, a0, a1, a2, eps=1e-14):
    """
    Exact value of

        K(t) = int_1^t ds / (a0 s^2 + a1 s + a2)

    for the all-nonzero quadratic case.
    """
    t = np.asarray(t, dtype=float)
    Delta = 4.0 * a0 * a2 - a1**2

    if Delta > eps:
        root = math.sqrt(Delta)
        return 2.0 / root * (
            np.arctan((2.0 * a0 * t + a1) / root)
            - math.atan((2.0 * a0 + a1) / root)
        )

    if abs(Delta) <= eps:
        return 2.0 * (
            1.0 / (2.0 * a0 + a1)
            - 1.0 / (2.0 * a0 * t + a1)
        )

    root = math.sqrt(-Delta)
    ratio_t = (2.0 * a0 * t + a1 - root) / (2.0 * a0 * t + a1 + root)
    ratio_1 = (2.0 * a0 + a1 - root) / (2.0 * a0 + a1 + root)
    return (1.0 / root) * _safe_log_abs(ratio_t / ratio_1)


def freise_L_from_t(p: Freise_SI_Params, t, eps: float = 1e-14):
    """
    Exact Freise first integral after t = exp(U):

        L(U) = int_1^t (s^2 - 1) / [s (a0 s^2 + a1 s + a2)] ds.

    The implementation follows the analytic case split from the derivation:
    a0 = 0, a2 = 0, a1 = 0, pure edge cases, and the general quadratic
    discriminant cases.
    """
    validate_params_si(p)
    t = np.asarray(t, dtype=float)
    scalar_input = np.ndim(t) == 0

    if np.any(t <= 0.0):
        raise ValueError("t = exp(U) must be positive.")

    a0, a1, a2 = freise_quadratic_coefficients(p)

    a0_zero = abs(a0) <= eps
    a1_zero = abs(a1) <= eps
    a2_zero = abs(a2) <= eps

    if a0_zero and a1_zero and a2_zero:
        raise ValueError("Invalid Freise coefficients: a0 = a1 = a2 = 0.")

    # P(t) = 1. This includes a0 = a1 = 0, a2 = 1.
    if a0_zero and a1_zero:
        L = 0.5 * (t**2 - 1.0) - np.log(t)

    # P(t) = t^2. This includes a1 = a2 = 0, a0 = 1.
    elif a1_zero and a2_zero:
        L = np.log(t) + 0.5 * (t**-2 - 1.0)

    # P(t) = t. This includes a0 = a2 = 0, a1 = 1.
    elif a0_zero and a2_zero:
        L = t + 1.0 / t - 2.0

    # a0 = 0, P(t) = a1 t + a2, with a1 + a2 = 1.
    elif a0_zero:
        P = a1 * t + a2
        L = (
            (a1 * t - a2 * np.log(P) + a2 - 1.0) / a1**2
            + np.log(P / t) / a2
        )

    # a2 = 0, P(t) = a0 t^2 + a1 t, with a0 + a1 = 1.
    elif a2_zero:
        P_over_t = a0 * t + a1
        L = (
            np.log(P_over_t) / a0
            + 1.0 / (a1 * t)
            - 1.0 / a1
            - (a0 / a1**2) * np.log(P_over_t / t)
        )

    # a1 = 0, P(t) = a0 t^2 + a2, with a0 + a2 = 1.
    elif a1_zero:
        P = a0 * t**2 + a2
        L = -np.log(t) / a2 + np.log(P) / (2.0 * a0 * a2)

    # General quadratic case.
    else:
        P = a0 * t**2 + a1 * t + a2
        K = _freise_quadratic_integral_K(t, a0, a1, a2, eps=eps)
        L = (
            -np.log(t) / a2
            + (a0 + a2) / (2.0 * a0 * a2) * np.log(P)
            + a1 * (a0 - a2) / (2.0 * a0 * a2) * K
        )

    # Remove tiny roundoff around U = 0.
    L = np.where(np.abs(L) < 1e-13, 0.0, L)

    if np.any(L < -1e-11):
        raise ValueError(
            "Exact Freise first integral became negative. "
            "This usually indicates an inadmissible parameter set or loss of precision."
        )

    L = np.maximum(L, 0.0)
    return float(L) if scalar_input else L


def freise_first_integral_U(p: Freise_SI_Params, U: float, n_quad: int = 800) -> float:
    """
    Exact Freise first integral.

    n_quad is kept only for backward compatibility with older notebook cells;
    it is not used anymore.
    """
    validate_params_si(p)
    U = float(U)
    if abs(U) < p.tol:
        return 0.0
    return float(freise_L_from_t(p, math.exp(U)))


def freise_first_integral_array(p: Freise_SI_Params, U_values, n_quad: int = 800):
    U_arr = np.asarray(U_values, dtype=float)
    t_arr = np.exp(U_arr)
    return freise_L_from_t(p, t_arr)


def sigma_free_F(p: Freise_SI_Params, V: float) -> float:
    validate_params_si(p)

    if abs(V) < p.tol:
        return 0.0

    U = V_to_U(p, V)
    beta = compute_beta(p)
    n_bulk = compute_n_bulk(p)

    L = freise_first_integral_U(p, U)
    prefactor = math.sqrt(2.0 * eps0 * p.eps_S * n_bulk / beta)

    return math.copysign(prefactor * math.sqrt(L), V)


def C_F(p: Freise_SI_Params, V: float) -> float:
    validate_params_si(p)

    beta = compute_beta(p)
    n_bulk = compute_n_bulk(p)
    prefactor = e * math.sqrt(beta * n_bulk * eps0 * p.eps_S / 2.0)

    # PZC limit.
    if abs(V) < p.tol:
        return 2.0 * prefactor

    U = V_to_U(p, V)
    t = math.exp(U)
    L = freise_first_integral_U(p, U)

    if L <= 0.0:
        return 2.0 * prefactor

    a0, a1, a2 = freise_quadratic_coefficients(p)
    P = a0 * t**2 + a1 * t + a2

    if P <= 0.0:
        raise ValueError(
            "Freise quadratic denominator became non-positive. "
            "Try smaller |phi|, smaller r0, smaller gamma values, or lower concentration."
        )

    return prefactor * math.copysign(1.0, V) * (t**2 - 1.0) / (P * math.sqrt(L))


# Solution scheme

This solution is based on rewriting the Freise equation as a first-order equation for the potential profile.  In dimensionless form,
$$
U = \frac{e_{0}\phi}{k_{\mathrm{B}}T},
\qquad
X = \frac{x}{\lambda_{\mathrm{D}}},
\qquad
\alpha = r_0^3N_Ac_b,
$$
and the Freise denominator is
$$
D(U)=1+\alpha\left[\gamma_c\left(e^{-U}-1\right)+\gamma_a\left(e^{U}-1\right)\right].
$$

The first integral can be written as
$$
\begin{aligned}
\left|\frac{\mathrm{d}U}{\mathrm{d}X}\right|
&= \sqrt{F(U)},\\
F(U) &= 2\int_0^U \frac{\sinh u}{D(u)}\,\mathrm{d}u .
\end{aligned}
$$

The code uses the exact Freise first integral from the previous section to evaluate $F(U)$, then solves the profile by inverse integration,
$$
X(U)=\int_U^{U_M}\frac{\mathrm{d}u}{\sqrt{F(u)}} .
$$

The resulting tabulated profile is interpolated onto the requested spatial grid.  Beyond the numerically integrated region, where the potential is already small, the code attaches a Debye--Hückel exponential tail instead of cutting the potential abruptly to zero.


In [ ]:
import math
import numpy as np


def freise_dU_dX_abs(p: Freise_SI_Params, U):
    U_arr = np.asarray(U, dtype=float)
    F = freise_first_integral_array(p, U_arr)
    return np.sqrt(F)


def freise_g_U(p: Freise_SI_Params, U):
    U_arr = np.asarray(U, dtype=float)
    val = freise_dU_dX_abs(p, U_arr)

    if np.any(val == 0.0):
        raise ZeroDivisionError("g(U) is singular at U = 0.")

    return 1.0 / val


def solve_U_profile_from_UM(
    UM: float,
    p: Freise_SI_Params,
    eps_U: float = 1e-6,
    max_points: int = 8000,
):
    validate_params_si(p)

    if eps_U <= 0.0:
        raise ValueError("eps_U must be > 0.")

    if abs(UM) <= eps_U:
        return (
            np.array([0.0]),
            np.array([0.0]),
            {
                "n_points": 1,
                "U_min": 0.5 * eps_U,
                "dU": 0.0,
                "X_max": 0.0,
                "message": "UM is already within the requested bulk tolerance.",
            },
        )

    sign = math.copysign(1.0, UM)
    U_end = sign * eps_U

    n_steps = min(max_points, max(500, int(900 * abs(UM))))
    U_grid = np.linspace(UM, U_end, n_steps + 1)

    F_vals = freise_first_integral_array(p, U_grid, n_quad=500)
    g_vals = 1.0 / np.sqrt(F_vals)

    dU_grid = np.abs(np.diff(U_grid))
    X_grid = np.zeros_like(U_grid)
    X_grid[1:] = np.cumsum(
        0.5 * (g_vals[:-1] + g_vals[1:]) * dU_grid
    )

    info = {
        "n_points": len(U_grid),
        "U_s": abs(UM),
        "U_min": abs(U_end),
        "dU": float(np.max(dU_grid)) if dU_grid.size else 0.0,
        "X_max": X_grid[-1],
        "alpha": compute_volume_parameter(p),
    }

    return X_grid, U_grid, info


def psi_Freise(
    x,
    p: Freise_SI_Params,
    V_M: float,
    phi_pzc: float = 0.0,
    eps_phi: float = 1e-6,
):
    validate_params_si(p)

    x_arr = np.asarray(x, dtype=float)
    scalar_input = np.ndim(x) == 0

    phi_s = V_M - phi_pzc

    if abs(phi_s) <= eps_phi:
        out = np.zeros_like(x_arr, dtype=float)
        return out.item() if scalar_input else out

    UM = V_to_U(p, phi_s)
    eps_U = abs(V_to_U(p, eps_phi))

    X_grid, U_grid, info = solve_U_profile_from_UM(
        UM,
        p,
        eps_U=eps_U,
        max_points=8000,
    )

    lam_D = compute_lambda_D(p)
    x_profile = X_grid * lam_D
    V_profile = U_to_V(p, U_grid)

    # Interpolate inside the computed profile.
    # For x > x_profile[-1], attach a Debye--Hückel tail
    # rather than imposing a hard cutoff to zero.
    psi = np.interp(
        x_arr,
        x_profile,
        V_profile,
        left=V_profile[0],
        right=np.nan,
    )

    tail_mask = np.isnan(psi)
    if np.any(tail_mask):
        x_tail = x_arr[tail_mask]
        psi[tail_mask] = V_profile[-1] * np.exp(
            -(x_tail - x_profile[-1]) / lam_D
        )

    return psi.item() if scalar_input else psi


def freise_denominator_from_psi(p: Freise_SI_Params, psi):
    """
    Freise denominator for symmetric 1:1 electrolyte:

        D = 1 + r0^3 n_b [gamma_c (exp(-U)-1) + gamma_a (exp(+U)-1)]

    where U = e psi / kBT.
    """
    psi_arr = np.asarray(psi, dtype=float)
    U = V_to_U(p, psi_arr)
    return freise_D_from_U(p, U)


def cation_concentration_Freise(x, p: Freise_SI_Params, psi):
    """
    Cation concentration for the symmetric 1:1 Freise model.

    Units are the same as p.c_bulk, i.e. mol/m^3.
    """
    psi_arr = np.asarray(psi, dtype=float)
    U = V_to_U(p, psi_arr)
    D = freise_denominator_from_psi(p, psi_arr)

    return p.c_bulk * np.exp(-U) / D


def anion_concentration_Freise(x, p: Freise_SI_Params, psi):
    """
    Anion concentration for the symmetric 1:1 Freise model.

    Units are the same as p.c_bulk, i.e. mol/m^3.
    """
    psi_arr = np.asarray(psi, dtype=float)
    U = V_to_U(p, psi_arr)
    D = freise_denominator_from_psi(p, psi_arr)

    return p.c_bulk * np.exp(+U) / D


# Interactive Freise sweep

The cells below mirror the interactive setup from the Gouy--Chapman, Gouy--Chapman--Stern, and Bikerman notebooks, but use the Freise finite-size diffuse-layer model. The controls sweep one physical input while the remaining parameters stay fixed at their base values.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C_PER_M2_TO_UC_PER_CM2 = 100.0
F_PER_M2_TO_UF_PER_CM2 = 100.0


def make_freise_params_and_VM_from_input(input_name, value, base_params, base_VM):
    r0 = base_params.r0
    gamma_c = base_params.gamma_c
    gamma_a = base_params.gamma_a
    eps_S = base_params.eps_S
    c_bulk = base_params.c_bulk
    T = base_params.T
    V_M = base_VM

    if input_name == "Concentration":
        if value <= 0:
            raise ValueError("Concentration sweep values must be > 0 mM")
        c_bulk = value

    elif input_name == "Temperature":
        if value <= 0:
            raise ValueError("Temperature sweep values must be > 0 K")
        T = value

    elif input_name == "Bulk dielectric constant":
        if value <= 0:
            raise ValueError("Bulk dielectric constant sweep values must be > 0")
        eps_S = value

    elif input_name == "Reference size":
        if value <= 0:
            raise ValueError("Reference size sweep values must be > 0 nm")
        r0 = value * 1e-9

    elif input_name == "Cation volume factor":
        if value < 0:
            raise ValueError("Cation volume factor sweep values must be >= 0")
        gamma_c = value

    elif input_name == "Anion volume factor":
        if value < 0:
            raise ValueError("Anion volume factor sweep values must be >= 0")
        gamma_a = value

    elif input_name == "Applied Potential":
        V_M = value

    else:
        raise ValueError(f"Unsupported input_name: {input_name}")

    p = Freise_SI_Params(
        r0=r0,
        gamma_c=gamma_c,
        gamma_a=gamma_a,
        eps_S=eps_S,
        c_bulk=c_bulk,
        T=T,
        tol=base_params.tol,
        maxiter=base_params.maxiter,
    )
    validate_params_si(p)
    return p, V_M


def _freise_label(input_name, val):
    if input_name == "Concentration":
        return f"{val:g} mM"
    if input_name == "Temperature":
        return f"{val:g} K"
    if input_name == "Applied Potential":
        return f"{val:g} V"
    if input_name == "Bulk dielectric constant":
        return rf"$\varepsilon_S = {val:g}$"
    if input_name == "Reference size":
        return rf"$r_0 = {val:g}$ nm"
    if input_name == "Cation volume factor":
        return rf"$\gamma_c = {val:g}$"
    if input_name == "Anion volume factor":
        return rf"$\gamma_a = {val:g}$"
    return f"{input_name} = {val:g}"


def plot_freise_profiles_sigma_cap(
    input_name,
    input_values,
    V_M,
    x_max_nm,
    npts,
    base_concentration,
    base_temperature,
    base_eps_S,
    base_r0_nm,
    base_gamma_c,
    base_gamma_a,
    phi_min,
    phi_max,
    sigma_npts=250,
):
    if phi_min >= phi_max:
        raise ValueError("phi_min must be smaller than phi_max")
    if x_max_nm <= 0:
        raise ValueError("x_max_nm must be > 0")
    if npts < 3:
        raise ValueError("npts must be at least 3")
    if base_r0_nm <= 0:
        raise ValueError("base r0 / reference size must be > 0 nm")
    if base_gamma_c < 0 or base_gamma_a < 0:
        raise ValueError("base gamma values must be >= 0")

    base_params = Freise_SI_Params(
        r0=base_r0_nm * 1e-9,
        gamma_c=base_gamma_c,
        gamma_a=base_gamma_a,
        eps_S=base_eps_S,
        c_bulk=base_concentration,
        T=base_temperature,
    )
    validate_params_si(base_params)

    x_nm = np.linspace(0.0, x_max_nm, npts)
    x_m = x_nm * 1e-9
    phi_grid = np.linspace(phi_min, phi_max, sigma_npts)

    fig = plt.figure(figsize=(12, 7), constrained_layout=True)
    gs = fig.add_gridspec(2, 6)

    ax1 = fig.add_subplot(gs[0, 0:2])
    ax2 = fig.add_subplot(gs[0, 2:4])
    ax3 = fig.add_subplot(gs[0, 4:6])
    ax4 = fig.add_subplot(gs[1, 0:3])
    ax5 = fig.add_subplot(gs[1, 3:6])

    sweeping_applied_potential = input_name == "Applied Potential"

    ncurves = len(input_values)
    tvals = np.array([0.0]) if ncurves == 1 else np.linspace(0.0, 1.0, ncurves, endpoint=False)

    def blend_with_white(rgb, strength=0.65):
        rgb = np.array(rgb[:3], dtype=float)
        return tuple((1 - strength) * np.ones(3) + strength * rgb)

    def mix(c1, c2, t):
        c1 = np.array(c1, dtype=float)
        c2 = np.array(c2, dtype=float)
        return tuple((1 - t) * c1 + t * c2)

    start_gray = np.array([0.15, 0.15, 0.15])
    end_gray = np.array([0.65, 0.65, 0.65])

    potential_colors = (
        [tuple(start_gray)]
        if ncurves == 1
        else [mix(start_gray, end_gray, i / (ncurves - 1)) for i in range(ncurves)]
    )
    cation_colors = [blend_with_white((1.0, 0.0, 0.0, 1.0), 0.25 + 0.65 * t) for t in tvals]
    anion_colors = [blend_with_white((0.0, 0.0, 1.0, 1.0), 0.25 + 0.65 * t) for t in tvals]
    sigma_colors = [blend_with_white((0.0, 0.55, 0.0, 1.0), 0.25 + 0.65 * t) for t in tvals]
    cap_colors = [blend_with_white((0.5, 0.0, 0.5, 1.0), 0.25 + 0.65 * t) for t in tvals]

    for i, val in enumerate(input_values):
        p, V_M_this = make_freise_params_and_VM_from_input(
            input_name=input_name,
            value=val,
            base_params=base_params,
            base_VM=V_M,
        )

        psi = psi_Freise(x_m, p, V_M_this)
        ncat = cation_concentration_Freise(x_m, p, psi)
        nani = anion_concentration_Freise(x_m, p, psi)

        label = _freise_label(input_name, val)

        ax1.plot(x_nm, psi, label=label, color=potential_colors[i], lw=2.5)
        ax2.plot(x_nm, ncat, label=label, color=cation_colors[i], lw=2)
        ax3.plot(x_nm, nani, label=label, color=anion_colors[i], lw=2)

        sigma_vals = C_PER_M2_TO_UC_PER_CM2 * np.array([sigma_free_F(p, V) for V in phi_grid])
        cap_vals = F_PER_M2_TO_UF_PER_CM2 * np.array([C_F(p, V) for V in phi_grid])

        ax4.plot(phi_grid, sigma_vals, label=label, color=sigma_colors[i], lw=2)
        ax5.plot(phi_grid, cap_vals, label=label, color=cap_colors[i], lw=2)

    ax1.set_title("Electric potential")
    ax1.set_xlabel("Distance from metal surface (nm)", fontsize=12)
    ax1.set_ylabel("Electric potential (V)", fontsize=12)

    ax2.set_title("Cation density")
    ax2.set_xlabel("Distance from metal surface (nm)", fontsize=12)
    ax2.set_ylabel("Cation density (mM = mol/m$^3$)", fontsize=12)

    ax3.set_title("Anion density")
    ax3.set_xlabel("Distance from metal surface (nm)", fontsize=12)
    ax3.set_ylabel("Anion density (mM = mol/m$^3$)", fontsize=12)

    ax4.set_title("Surface free charge density")
    ax4.set_xlabel(r"$\phi_M - \phi_{pzc}$ (V)", fontsize=14)
    ax4.set_ylabel(r"Surface free charge density ($\mu$C/cm$^2$)")

    ax5.set_title("Freise differential capacitance")
    ax5.set_xlabel(r"$\phi_M - \phi_{pzc}$ (V)", fontsize=14)
    ax5.set_ylabel(r"Double-layer capacitance ($\mu$F/cm$^2$)")

    for ax in (ax1, ax2, ax3, ax4, ax5):
        ax.grid(True)

    ax1.legend(fontsize=8, loc="lower left")
    ax2.legend(fontsize=8, loc="lower left")
    ax3.legend(fontsize=8, loc="lower left")

    if not sweeping_applied_potential:
        ax4.legend(fontsize=9)
        ax5.legend(fontsize=9)

    return fig


# Define error box


In [ ]:
import ipywidgets as widgets


def show_error(message):
    display(widgets.HTML(
        f"""
        <div style="
            color:#8a1f11;
            background:#fff3f0;
            border:1px solid #f0b4a8;
            border-radius:6px;
            padding:10px;
            font-size:14px;
            max-width:700px;
        ">
            <b>Error:</b> {message}
        </div>
        """
    ))


# Make User Interface




In [ ]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

LABEL_WIDTH = "210px"
BOX_WIDTH = "150px"


def make_row(label_html, widget, fontsize="16px"):
    widget.layout = widgets.Layout(width=BOX_WIDTH)
    label = widgets.HTML(
        value=f"""
        <span style="font-size:{fontsize}; line-height:1.2;">
            {label_html}
        </span>
        """,
        layout=widgets.Layout(width=LABEL_WIDTH),
    )
    return widgets.HBox([label, widget], layout=widgets.Layout(align_items="center"))


input_dropdown = widgets.Dropdown(
    options=[
        "Concentration",
        "Temperature",
        "Bulk dielectric constant",
        "Reference size",
        "Cation volume factor",
        "Anion volume factor",
        "Applied Potential",
    ],
    value="Anion volume factor",
    layout=widgets.Layout(width=BOX_WIDTH),
)
input_row = make_row("Input:", input_dropdown)

values_text = widgets.Text(value="0.5, 1, 2", placeholder="comma-separated values")
values_row = make_row("Sweep:", values_text)

VM_box = widgets.FloatText(value=0.2)
VM_row = make_row("ϕ<sub>M</sub> − ϕ<sub>pzc</sub> (V):", VM_box)

phi_min_box = widgets.FloatText(value=-0.3)
phi_min_row = make_row("ϕ<sub>min</sub> (V):", phi_min_box)

phi_max_box = widgets.FloatText(value=0.3)
phi_max_row = make_row("ϕ<sub>max</sub> (V):", phi_max_box)

xmax_box = widgets.FloatText(value=5.0)
xmax_row = make_row("x<sub>max</sub> (nm):", xmax_box)

base_c_box = widgets.FloatText(value=10.0)
base_c_row = make_row("base c (mM):", base_c_box)

base_T_box = widgets.FloatText(value=298.15)
base_T_row = make_row("base T (K):", base_T_box)

base_eps_S_box = widgets.FloatText(value=78.5)
base_eps_S_row = make_row("base ε<sub>S</sub>:", base_eps_S_box)

base_r0_box = widgets.FloatText(value=1.3)
base_r0_row = make_row("base r<sub>0</sub> / ref. size (nm):", base_r0_box)

base_gamma_c_box = widgets.FloatText(value=1.0)
base_gamma_c_row = make_row("base γ<sub>c</sub>:", base_gamma_c_box)

base_gamma_a_box = widgets.FloatText(value=1.0)
base_gamma_a_row = make_row("base γ<sub>a</sub>:", base_gamma_a_box)

plot_button = widgets.Button(
    description="Plot",
    button_style="success",
    layout=widgets.Layout(width="120px"),
)
plot_row = widgets.HBox([widgets.HTML(layout=widgets.Layout(width=LABEL_WIDTH)), plot_button])

out = widgets.Output(layout=widgets.Layout(width="auto", border="none"))
npts = 300

base_header = widgets.HTML(
    """
    <div style="
        font-size:14px;
        font-weight:600;
        margin-top:8px;
        margin-bottom:4px;
    ">
        Base values used for parameters not being swept:
    </div>
    """
)


def update_visible_base_inputs(*args):
    swept = input_dropdown.value

    base_c_row.layout.display = "" if swept != "Concentration" else "none"
    base_T_row.layout.display = "" if swept != "Temperature" else "none"
    base_eps_S_row.layout.display = "" if swept != "Bulk dielectric constant" else "none"
    base_r0_row.layout.display = "" if swept != "Reference size" else "none"
    base_gamma_c_row.layout.display = "" if swept != "Cation volume factor" else "none"
    base_gamma_a_row.layout.display = "" if swept != "Anion volume factor" else "none"

    VM_row.layout.display = "none" if swept == "Applied Potential" else ""


def on_plot_clicked(b):
    out.clear_output(wait=True)
    plt.close("all")

    try:
        input_values = np.array(
            [float(v.strip()) for v in values_text.value.split(",") if v.strip()]
        )

        if input_values.size == 0:
            raise ValueError("Please enter at least one sweep value.")
        if xmax_box.value <= 0:
            raise ValueError("x max must be > 0 nm.")
        if npts < 3:
            raise ValueError("Points must be at least 3.")
        if phi_min_box.value >= phi_max_box.value:
            raise ValueError("phi min must be smaller than phi max.")
        if base_r0_box.value <= 0:
            raise ValueError("base r0 / reference size must be > 0 nm.")
        if base_gamma_c_box.value < 0 or base_gamma_a_box.value < 0:
            raise ValueError("base gamma values must be >= 0.")

        fig = plot_freise_profiles_sigma_cap(
            input_name=input_dropdown.value,
            input_values=input_values,
            V_M=VM_box.value,
            x_max_nm=xmax_box.value,
            npts=npts,
            base_concentration=base_c_box.value,
            base_temperature=base_T_box.value,
            base_eps_S=base_eps_S_box.value,
            base_r0_nm=base_r0_box.value,
            base_gamma_c=base_gamma_c_box.value,
            base_gamma_a=base_gamma_a_box.value,
            phi_min=phi_min_box.value,
            phi_max=phi_max_box.value,
            sigma_npts=npts,
        )

        with out:
            display(fig)
        plt.close(fig)

    except ValueError as err:
        plt.close("all")
        out.clear_output(wait=True)
        with out:
            show_error(f"Invalid input: {err}")

    except Exception as err:
        plt.close("all")
        out.clear_output(wait=True)
        with out:
            show_error(f"Unexpected error: {err}")


try:
    input_dropdown.unobserve(update_visible_base_inputs, names="value")
except Exception:
    pass

input_dropdown.observe(update_visible_base_inputs, names="value")

plot_button.on_click(on_plot_clicked, remove=True)
plot_button.on_click(on_plot_clicked)

update_visible_base_inputs()

controls = widgets.VBox(
    [
        input_row,
        values_row,
        VM_row,
        phi_min_row,
        phi_max_row,
        xmax_row,
        base_header,
        base_c_row,
        base_T_row,
        base_eps_S_row,
        base_r0_row,
        base_gamma_c_row,
        base_gamma_a_row,
        plot_row,
    ],
    layout=widgets.Layout(width="390px", min_width="390px"),
)

output_panel = widgets.Box(
    [out],
    layout=widgets.Layout(flex="1 1 0%", width="auto"),
)

ui = widgets.HBox(
    [controls, output_panel],
    layout=widgets.Layout(width="100%", align_items="flex-start"),
)

display(ui)


# Compare with Valette experimental data

The Freise comparison below uses the same $\gamma_{\mathrm{c}}$ and $\gamma_{\mathrm{a}}$ parameters as the model curves. These enter the denominator

$$
D(U)=1+r_0^3N_Ac_b\left[\gamma_{\mathrm{c}}(e^{-U}-1)+\gamma_{\mathrm{a}}(e^{U}-1)\right].
$$


In [ ]:
from pathlib import Path
import numpy as np

VALETTE_FILES = {
    5.0: "Valette1982_5mM.txt",
    10.0: "Valette1982_10mM.txt",
    20.0: "Valette1982_20mM.txt",
    40.0: "Valette1982_40mM.txt",
    100.0: "Valette1982_100mM.txt",
}


def load_valette_txt(filepath):
    E_vals = []
    C_vals = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue
            if line.startswith("%") or line.startswith("#"):
                continue

            parts = line.replace(",", " ").split()
            if len(parts) < 2:
                continue

            try:
                E_vals.append(float(parts[0]))
                C_vals.append(float(parts[1]))
            except ValueError:
                # Allow a small amount of header text.
                continue

    if not E_vals:
        raise ValueError(f"No numeric data found in {filepath}")

    return np.array(E_vals, dtype=float), np.array(C_vals, dtype=float)


def find_valette_data_dir():
    """
    Try the same local folder layout as the Bikerman and Gouy--Chapman--Stern
    notebooks, while also allowing the data folder to sit next to this notebook.
    """
    candidates = [
        Path.cwd() / "Valette1982",
        Path.cwd().parent / "Valette1982",
        Path.cwd(),
        Path.cwd().parent,
        Path("/mnt/data/Valette1982"),
        Path("/mnt/data"),
    ]

    for directory in candidates:
        if all((directory / filename).exists() for filename in VALETTE_FILES.values()):
            return directory

    for directory in candidates:
        if any((directory / filename).exists() for filename in VALETTE_FILES.values()):
            return directory

    return candidates[0]


DATA_DIR = find_valette_data_dir()


def load_all_valette_data(data_dir=DATA_DIR, quiet=False):
    data = {}
    missing = []

    for conc_mM, filename in VALETTE_FILES.items():
        filepath = Path(data_dir) / filename

        if filepath.exists():
            E, C = load_valette_txt(filepath)
            data[conc_mM] = {"E": E, "C": C, "path": filepath}
        else:
            missing.append(filepath)

    if missing and not quiet:
        print("Missing Valette files:")
        for filepath in missing:
            print(f"  {filepath}")

    return data, missing


VALETTE_DATA, VALETTE_MISSING = load_all_valette_data()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def make_freise_params_for_valette(
    concentration_mM,
    temperature,
    eps_S,
    r0_nm,
    gamma_c,
    gamma_a,
):
    # Keep the experimental-data comparison on the same Freise
    # parameterization as the model UI. The Valette curves therefore
    # receive both ion-volume asymmetry factors, gamma_c and gamma_a.
    p = Freise_SI_Params(
        r0=r0_nm * 1e-9,
        gamma_c=float(gamma_c),
        gamma_a=float(gamma_a),
        eps_S=eps_S,
        c_bulk=concentration_mM,
        T=temperature,
    )
    validate_params_si(p)
    return p


def plot_freise_valette_comparison(
    concentrations_mM,
    temperature=298.15,
    eps_S=78.5,
    r0_nm=1.3,
    gamma_c=1.0,
    gamma_a=1.0,
    phi_pzc_sce=-0.92,
    E_min=-1.48,
    E_max=-0.32,
    npts=350,
    show_data=True,
    show_missing_note=True,
):

    if E_min >= E_max:
        raise ValueError("E min must be smaller than E max.")
    if r0_nm <= 0:
        raise ValueError("Reference size r0 must be > 0 nm.")
    if gamma_c < 0 or gamma_a < 0:
        raise ValueError("gamma values must be >= 0.")
    if eps_S <= 0:
        raise ValueError("Bulk/solution dielectric constant ε_S must be > 0.")
    if temperature <= 0:
        raise ValueError("Temperature must be > 0 K.")
    if npts < 3:
        raise ValueError("npts must be at least 3.")

    concentrations_mM = [float(c) for c in concentrations_mM]
    if len(concentrations_mM) == 0:
        raise ValueError("Please enter at least one concentration.")

    E_grid = np.linspace(E_min, E_max, npts)
    V_model = E_grid - phi_pzc_sce

    fig = plt.figure(figsize=(12, 8.5), constrained_layout=True)
    gs = fig.add_gridspec(1, 1)
    ax = fig.add_subplot(gs[0, 0])

    cmap = plt.cm.magma_r
    ncurves = len(concentrations_mM)
    colors = [cmap(0.65)] if ncurves == 1 else [cmap(t) for t in np.linspace(0.15, 0.90, ncurves)]

    for color, conc in zip(colors, concentrations_mM):
        p = make_freise_params_for_valette(
            concentration_mM=conc,
            temperature=temperature,
            eps_S=eps_S,
            r0_nm=r0_nm,
            gamma_c=gamma_c,
            gamma_a=gamma_a,
        )

        cap_model = F_PER_M2_TO_UF_PER_CM2 * np.array(
            [C_F(p, V) for V in V_model]
        )

        ax.plot(
            E_grid,
            cap_model,
            lw=2.6,
            color=color,
            solid_capstyle="round",
            label=(
                rf"Freise {conc:g} mM "
                rf"($\gamma_c$={gamma_c:g}, $\gamma_a$={gamma_a:g})"
            ),
        )

        if show_data and conc in VALETTE_DATA:
            data = VALETTE_DATA[conc]
            darker = tuple(np.clip(np.array(color[:3]) * 0.72, 0, 1))

            scatter_kwargs = dict(
                s=36,
                marker="o",
                facecolor=darker,
                edgecolor="white",
                linewidth=0.7,
                alpha=0.95,
                zorder=3,
            )

            ax.scatter(
                data["E"],
                data["C"],
                label=rf"Valette 1982 {conc:g} mM",
                **scatter_kwargs,
            )

    ax.axvline(phi_pzc_sce, color="0.35", lw=1.2, ls="--")
    ax.set_title(
        rf"Freise capacitance compared with Valette 1982 "
        rf"($\gamma_c$={gamma_c:g}, $\gamma_a$={gamma_a:g})",
        fontsize=18,
        pad=12,
    )
    ax.set_xlabel(r"ϕ$_{\mathrm{M}}$ (V)", fontsize=14)
    ax.set_ylabel(r"Differential capacitance ($\mu$F/cm$^2$)", fontsize=14)

    ax.grid(True, alpha=0.25)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=11)

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(
            fontsize=8.5,
            frameon=True,
            fancybox=True,
            framealpha=0.92,
            edgecolor="0.85",
            ncols=2,
        )

    if show_missing_note and show_data and VALETTE_MISSING:
        available = ", ".join(f"{c:g} mM" for c in sorted(VALETTE_DATA))
        if available:
            note = f"Loaded Valette data: {available}."
        else:
            note = (
                "No Valette data files were found. "
                "Run the Valette data-loading cell and check the Google Drive folder."
            )

        fig.text(
            0.01,
            0.01,
            note,
            ha="left",
            va="bottom",
            fontsize=9,
            color="0.35",
        )

    return fig


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

VAL_LABEL_WIDTH = "190px"
VAL_BOX_WIDTH = "150px"


def make_valette_row(label_html, widget, fontsize="16px"):
    widget.layout = widgets.Layout(width=VAL_BOX_WIDTH)
    label = widgets.HTML(
        value=f"""
        <span style="font-size:{fontsize}; line-height:1.2;">
            {label_html}
        </span>
        """,
        layout=widgets.Layout(width=VAL_LABEL_WIDTH),
    )
    return widgets.HBox([label, widget], layout=widgets.Layout(align_items="center"))


valette_T_box = widgets.FloatText(value=298.15)
valette_T_row = make_valette_row("T (K):", valette_T_box)

valette_eps_S_box = widgets.FloatText(value=78.5)
valette_eps_S_row = make_valette_row("ε<sub>S</sub>:", valette_eps_S_box)

valette_r0_box = widgets.FloatText(value=0.6)
valette_r0_row = make_valette_row("r<sub>0</sub> / ref. size (nm):", valette_r0_box)

valette_gamma_c_box = widgets.FloatText(value=20.0)
valette_gamma_c_row = make_valette_row("γ<sub>c</sub>:", valette_gamma_c_box)

valette_gamma_a_box = widgets.FloatText(value=17.0)
valette_gamma_a_row = make_valette_row("γ<sub>a</sub>:", valette_gamma_a_box)

valette_phi_pzc_box = widgets.FloatText(value=-0.92)
valette_phi_pzc_row = make_valette_row("ϕ<sub>pzc</sub> (V):", valette_phi_pzc_box)

valette_E_min_box = widgets.FloatText(value=-1.48)
valette_E_min_row = make_valette_row("ϕ<sub>min</sub> (V):", valette_E_min_box)

valette_E_max_box = widgets.FloatText(value=-0.32)
valette_E_max_row = make_valette_row("ϕ<sub>max</sub> (V):", valette_E_max_box)

valette_plot_button = widgets.Button(
    description="Plot comparison",
    button_style="success",
    layout=widgets.Layout(width="150px"),
)

valette_plot_row = widgets.HBox(
    [widgets.HTML(layout=widgets.Layout(width=VAL_LABEL_WIDTH)), valette_plot_button]
)

valette_out = widgets.Output(layout=widgets.Layout(width="auto", border="none"))


def on_valette_freise_plot_clicked(b):
    valette_out.clear_output(wait=True)
    plt.close("all")

    try:
        concentrations = [5.0, 10.0, 20.0, 40.0, 100.0]

        if valette_T_box.value <= 0:
            raise ValueError("T must be > 0 K.")
        if valette_eps_S_box.value <= 0:
            raise ValueError("ε_S must be > 0.")
        if valette_r0_box.value <= 0:
            raise ValueError("r0 / reference size must be > 0 nm.")
        if valette_gamma_c_box.value < 0 or valette_gamma_a_box.value < 0:
            raise ValueError("gamma values must be >= 0.")
        if valette_E_min_box.value >= valette_E_max_box.value:
            raise ValueError("ϕ_min must be smaller than ϕ_max.")

        fig = plot_freise_valette_comparison(
            concentrations_mM=concentrations,
            temperature=valette_T_box.value,
            eps_S=valette_eps_S_box.value,
            r0_nm=valette_r0_box.value,
            gamma_c=valette_gamma_c_box.value,
            gamma_a=valette_gamma_a_box.value,
            phi_pzc_sce=valette_phi_pzc_box.value,
            E_min=valette_E_min_box.value,
            E_max=valette_E_max_box.value,
            npts=350,
        )

        with valette_out:
            display(fig)
        plt.close(fig)

    except ValueError as err:
        plt.close("all")
        valette_out.clear_output(wait=True)
        with valette_out:
            show_error(f"Invalid input: {err}")

    except Exception as err:
        plt.close("all")
        valette_out.clear_output(wait=True)
        with valette_out:
            show_error(f"Unexpected error: {err}")


try:
    valette_plot_button.on_click(on_valette_freise_plot_clicked, remove=True)
except Exception:
    pass

valette_plot_button.on_click(on_valette_freise_plot_clicked)

valette_controls = widgets.VBox(
    [
        valette_T_row,
        valette_eps_S_row,
        valette_r0_row,
        valette_gamma_c_row,
        valette_gamma_a_row,
        valette_phi_pzc_row,
        valette_E_min_row,
        valette_E_max_row,
        valette_plot_row,
    ],
    layout=widgets.Layout(width="390px", min_width="390px"),
)

valette_output_panel = widgets.Box(
    [valette_out],
    layout=widgets.Layout(flex="1 1 0%", width="auto"),
)

valette_ui = widgets.HBox(
    [valette_controls, valette_output_panel],
    layout=widgets.Layout(width="100%", align_items="flex-start"),
)

display(valette_ui)
